# FINANCE 384 Assignment 1 – Part A

## A.6 Economic Performance

This notebook translates the return forecasts into the assignment's required prediction-sorted quintile portfolios.

For each valid decision month \(t\) and each model:

1. sort stocks from lowest to highest predicted next-month excess return;
2. assign the \(k\)-th sorted stock among \(N_t\) stocks to

\[
q_{k,t}=1+\left\lfloor \frac{5(k-1)}{N_t}\right\rfloor;
\]

3. calculate equal-weighted realised excess returns for P1–P5;
4. form the monthly long-short spread \(P5-P1\);
5. report the average monthly \(P5-P1\) excess return and its t-statistic;
6. estimate monthly market-model alpha:

\[
r^{p,e}_{t+1}=\alpha_p+\beta_p MKT_{t+1}+u^p_{t+1}.
\]

A portfolio formed using information at month \(t\) is matched to `mktrf` realised in month \(t+1\).

Required model-period combinations:

- Random Forest — validation
- Random Forest — test
- Pooled OLS — test


### A.6 modelling protocol

- **Random Forest:** A.4-selected specification, fitted on training only
- **Pooled OLS:** fitted on training + validation
- **Portfolio weighting:** equal weight within each quintile
- **Quintile rule:** exact assignment formula specified in the assignment
- **Test comparison:** identical stock-month rows for Random Forest and OLS
- **Market attribution:** match decision month \(t\) to market return in holding month \(t+1\)
- **Return t-statistic:** sample mean divided by its standard error using monthly P5−P1 returns
- **Alpha t-statistic:** intercept t-statistic from the monthly market-model regression

When predicted returns are tied, stocks are ordered by `permno` only as a deterministic tie-breaker for reproducibility. `permno` is not used as a predictor.


In [ ]:
# A.6.1 Imports and settings

import pandas as pd
import numpy as np
import statsmodels.api as sm

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor

PANEL_FILE = "FINANCE384_assignmentA_development_panel.csv"
DICTIONARY_FILE = "FINANCE384_stock_month_data_dictionary.csv"
from pathlib import Path

MARKET_FILE = (
    "FINANCE384_market.csv"
    if Path("FINANCE384_market.csv").exists()
    else "FINANCE384_market(2).csv"
)

TARGET = "target_ret_excess_tp1"
RANDOM_SEED = 384

RF_PARAMS = {
    "n_estimators": 100,
    "max_depth": 2,
    "min_samples_leaf": 8,
    "max_features": 0.3,
    "random_state": RANDOM_SEED,
    "n_jobs": -1,
}


In [ ]:
# A.6.2 Load supplied data

panel = pd.read_csv(PANEL_FILE)
data_dictionary = pd.read_csv(DICTIONARY_FILE)
market = pd.read_csv(
    MARKET_FILE,
    parse_dates=["date"],
)

panel["date"] = pd.to_datetime(
    panel["date"]
)

market["holding_month"] = (
    market["date"]
    .dt.to_period("M")
)

print("Panel shape:", panel.shape)
print("Market shape:", market.shape)

print(
    "Market date range:",
    market["date"].min().date(),
    "to",
    market["date"].max().date(),
)

print(
    "Market months:",
    market["holding_month"].nunique(),
)

print(
    "Missing market returns:",
    int(market["mktrf"].isna().sum()),
)


In [ ]:
# A.6.3 Define the common revised A.1 predictor information

numeric_predictors = [
    "size",
    "bm",
    "mom12_2",
    "vol12",
    "beta60",
    "ivol60",
    "turnover",
    "dollar_volume",
    "amihud_illiq",
    "divyield",
    "gross_profit",
    "roe",
    "asset_growth",
    "leverage",
    "accruals",
    "mkt_12m",
    "mkt_vol_12m",
    "down_market",
]

continuous_predictors = [
    col for col in numeric_predictors
    if col != "down_market"
]

binary_predictors = ["down_market"]
categorical_predictors = ["ff49_code"]


In [ ]:
# A.6.4 Construct the next-calendar-month excess-return target

panel = (
    panel
    .sort_values(["permno", "date"])
    .reset_index(drop=True)
)

panel["month"] = (
    panel["date"]
    .dt.to_period("M")
)

next_month_return = (
    panel[
        ["permno", "month", "ret_excess_t"]
    ]
    .rename(
        columns={
            "ret_excess_t": TARGET
        }
    )
    .assign(
        month=lambda df: df["month"] - 1
    )
)

panel = panel.merge(
    next_month_return,
    on=["permno", "month"],
    how="left",
    validate="one_to_one",
)

print(
    "Rows with valid next-month target:",
    panel[TARGET].notna().sum(),
)


In [ ]:
# A.6.5 Create missingness indicators and impute characteristics

missing_characteristics = [
    col
    for col in continuous_predictors
    if panel[col].isna().any()
]

missing_indicator_columns = []

for col in missing_characteristics:
    indicator = f"{col}_was_missing"

    panel[indicator] = (
        panel[col]
        .isna()
        .astype(int)
    )

    missing_indicator_columns.append(
        indicator
    )

    industry_month_median = (
        panel
        .groupby(
            ["month", "ff49_code"]
        )[col]
        .transform("median")
    )

    market_month_median = (
        panel
        .groupby("month")[col]
        .transform("median")
    )

    panel[col] = (
        panel[col]
        .fillna(industry_month_median)
        .fillna(market_month_median)
    )

print(
    "Missingness indicators:",
    len(missing_indicator_columns),
)

print(
    "Remaining missing continuous values:",
    int(
        panel[continuous_predictors]
        .isna()
        .sum()
        .sum()
    ),
)


In [ ]:
# A.6.6 Apply prescribed chronological split

analysis = panel.loc[
    panel[TARGET].notna()
].copy()

train = analysis.loc[
    (analysis["date"] >= "1990-01-01")
    & (analysis["date"] <= "2014-12-31")
].copy()

validation = analysis.loc[
    (analysis["date"] >= "2015-01-01")
    & (analysis["date"] <= "2018-12-31")
].copy()

test = analysis.loc[
    (analysis["date"] >= "2019-01-01")
    & (analysis["date"] <= "2022-11-30")
].copy()

train_valid = (
    pd.concat(
        [train, validation],
        axis=0,
    )
    .sort_values(
        ["date", "permno"]
    )
    .reset_index(drop=True)
)

pd.DataFrame({
    "Sample": [
        "Training",
        "Validation",
        "Train + Validation",
        "Test",
    ],
    "Months": [
        train["month"].nunique(),
        validation["month"].nunique(),
        train_valid["month"].nunique(),
        test["month"].nunique(),
    ],
    "Rows": [
        len(train),
        len(validation),
        len(train_valid),
        len(test),
    ],
})


In [ ]:
# A.6.7 Build and transform the common feature matrices

raw_feature_columns = (
    continuous_predictors
    + binary_predictors
    + missing_indicator_columns
    + categorical_predictors
)

preprocessor = ColumnTransformer(
    transformers=[
        (
            "continuous",
            StandardScaler(),
            continuous_predictors,
        ),
        (
            "binary",
            "passthrough",
            binary_predictors
            + missing_indicator_columns,
        ),
        (
            "industry",
            OneHotEncoder(
                drop="first",
                handle_unknown="ignore",
                sparse_output=False,
            ),
            categorical_predictors,
        ),
    ],
    remainder="drop",
)

X_train_raw = train[
    raw_feature_columns
].copy()

X_validation_raw = validation[
    raw_feature_columns
].copy()

X_train_valid_raw = train_valid[
    raw_feature_columns
].copy()

X_test_raw = test[
    raw_feature_columns
].copy()

y_train = train[
    TARGET
].to_numpy()

y_validation = validation[
    TARGET
].to_numpy()

y_train_valid = train_valid[
    TARGET
].to_numpy()

y_test = test[
    TARGET
].to_numpy()

preprocessor.fit(
    X_train_raw
)

X_train = preprocessor.transform(
    X_train_raw
)

X_validation = preprocessor.transform(
    X_validation_raw
)

X_train_valid = preprocessor.transform(
    X_train_valid_raw
)

X_test = preprocessor.transform(
    X_test_raw
)

print("Training matrix:", X_train.shape)
print("Validation matrix:", X_validation.shape)
print("Train + validation matrix:", X_train_valid.shape)
print("Test matrix:", X_test.shape)


## Generate model forecasts

For this standalone A.6 notebook:

- pooled OLS is fitted on training + validation;
- the A.4-selected Random Forest specification is fitted on training only.

In the final merged notebook, the already retained A.4 Random Forest model should be carried forward directly.


In [ ]:
# A.6.8 Fit pooled OLS and selected Random Forest

ols_model = LinearRegression(
    fit_intercept=True
)

ols_model.fit(
    X_train_valid,
    y_train_valid,
)

selected_random_forest = RandomForestRegressor(
    **RF_PARAMS
)

selected_random_forest.fit(
    X_train,
    y_train,
)

rf_validation_pred = (
    selected_random_forest.predict(
        X_validation
    )
)

rf_test_pred = (
    selected_random_forest.predict(
        X_test
    )
)

ols_test_pred = (
    ols_model.predict(
        X_test
    )
)

print(
    "RF validation predictions:",
    len(rf_validation_pred),
)

print(
    "RF test predictions:",
    len(rf_test_pred),
)

print(
    "OLS test predictions:",
    len(ols_test_pred),
)


In [ ]:
# A.6.9 Assemble prediction datasets

rf_validation = validation[
    ["date", "permno", "ticker", TARGET]
].copy()

rf_validation["prediction"] = (
    rf_validation_pred
)

rf_test = test[
    ["date", "permno", "ticker", TARGET]
].copy()

rf_test["prediction"] = (
    rf_test_pred
)

ols_test = test[
    ["date", "permno", "ticker", TARGET]
].copy()

ols_test["prediction"] = (
    ols_test_pred
)

print(
    "RF validation rows:",
    len(rf_validation),
)

print(
    "RF test rows:",
    len(rf_test),
)

print(
    "OLS test rows:",
    len(ols_test),
)


In [ ]:
# A.6.10 Confirm common test rows

rf_test_keys = set(
    zip(
        rf_test["date"],
        rf_test["permno"],
    )
)

ols_test_keys = set(
    zip(
        ols_test["date"],
        ols_test["permno"],
    )
)

print(
    "Same RF and OLS test stock-month rows:",
    rf_test_keys == ols_test_keys,
)

print(
    "Common test rows:",
    len(rf_test_keys),
)


## Exact quintile construction

The function below implements the assignment's exact rule rather than `pd.qcut()`.

Within each month, observations are sorted from low to high predicted return. If forecasts are tied, `permno` is used only as a deterministic tie-breaker.

For sorted position \(k=1,\ldots,N_t\),

\[
q_{k,t}=1+\left\lfloor \frac{5(k-1)}{N_t}\right\rfloor.
\]

P1 therefore contains the lowest predicted returns and P5 the highest.


In [ ]:
# A.6.11 Exact prediction-sorted quintile function

def build_quintile_portfolios(
    prediction_df,
    prediction_col="prediction",
):
    ranked = prediction_df[
        ["date", "permno", TARGET, prediction_col]
    ].dropna().copy()

    ranked = (
        ranked
        .sort_values(
            ["date", prediction_col, "permno"],
            ascending=[True, True, True],
            kind="mergesort",
        )
        .reset_index(drop=True)
    )

    ranked["k"] = (
        ranked
        .groupby("date")
        .cumcount()
        + 1
    )

    ranked["N_t"] = (
        ranked
        .groupby("date")["permno"]
        .transform("size")
    )

    ranked["quintile"] = (
        1
        + np.floor(
            5
            * (ranked["k"] - 1)
            / ranked["N_t"]
        )
    ).astype(int)

    assert ranked[
        "quintile"
    ].between(1, 5).all()

    monthly_portfolios = (
        ranked
        .groupby(
            ["date", "quintile"]
        )[TARGET]
        .mean()
        .unstack("quintile")
        .sort_index()
    )

    monthly_portfolios.columns = [
        f"P{int(col)}"
        for col in monthly_portfolios.columns
    ]

    monthly_portfolios["P5-P1"] = (
        monthly_portfolios["P5"]
        - monthly_portfolios["P1"]
    )

    return ranked, monthly_portfolios


In [ ]:
# A.6.12 Build required portfolios

rf_validation_ranked, rf_validation_portfolios = (
    build_quintile_portfolios(
        rf_validation
    )
)

rf_test_ranked, rf_test_portfolios = (
    build_quintile_portfolios(
        rf_test
    )
)

ols_test_ranked, ols_test_portfolios = (
    build_quintile_portfolios(
        ols_test
    )
)

print(
    "RF validation portfolio months:",
    len(rf_validation_portfolios),
)

print(
    "RF test portfolio months:",
    len(rf_test_portfolios),
)

print(
    "OLS test portfolio months:",
    len(ols_test_portfolios),
)


### Quintile construction audit

Every valid month should contain all five portfolios. The test-period Random Forest and OLS sorts should also start from the same number of stocks each month because they use identical test rows.


In [ ]:
# A.6.13 Quintile assignment audit

def quintile_audit(ranked):
    counts = (
        ranked
        .groupby(
            ["date", "quintile"]
        )
        .size()
        .unstack(
            fill_value=0
        )
    )

    return pd.DataFrame({
        "minimum_stocks_per_quintile":
            counts.min(),
        "maximum_stocks_per_quintile":
            counts.max(),
    })

print("RF validation:")
display(
    quintile_audit(
        rf_validation_ranked
    )
)

print("RF test:")
display(
    quintile_audit(
        rf_test_ranked
    )
)

print("OLS test:")
display(
    quintile_audit(
        ols_test_ranked
    )
)


## Average P5−P1 return and t-statistic

For monthly spread returns \(x_t\), the reported t-statistic is

\[
t=\frac{\bar{x}}{s_x/\sqrt{T}},
\]

where \(s_x\) is the sample standard deviation using \(T-1\) degrees of freedom.


In [ ]:
# A.6.14 Mean-return t-statistic

def mean_return_stats(
    monthly_returns,
):
    x = (
        pd.Series(monthly_returns)
        .dropna()
        .astype(float)
    )

    mean_return = float(
        x.mean()
    )

    standard_error = float(
        x.std(ddof=1)
        / np.sqrt(len(x))
    )

    t_stat = float(
        mean_return
        / standard_error
    )

    return {
        "mean_p5_minus_p1":
            mean_return,
        "mean_return_t_stat":
            t_stat,
        "months":
            int(len(x)),
    }


## Market-model alpha

The market file records `mktrf` in the month in which the market return is realised.

Therefore, a portfolio formed at decision month \(t\) is assigned

\[
\text{holding month}=t+1
\]

before merging to `mktrf`.

The monthly regression is

\[
P5-P1_{t+1}
=
\alpha
+
\beta\,mktrf_{t+1}
+
u_{t+1}.
\]

The reported alpha is the monthly intercept, together with its conventional OLS t-statistic.


In [ ]:
# A.6.15 Market-model alpha function

def market_model_alpha(
    monthly_portfolios,
):
    spread = (
        monthly_portfolios[
            ["P5-P1"]
        ]
        .reset_index()
        .copy()
    )

    spread[
        "decision_month"
    ] = (
        spread["date"]
        .dt.to_period("M")
    )

    spread[
        "holding_month"
    ] = (
        spread["decision_month"]
        + 1
    )

    spread = spread.merge(
        market[
            ["holding_month", "mktrf"]
        ],
        on="holding_month",
        how="left",
        validate="one_to_one",
    )

    if spread[
        "mktrf"
    ].isna().any():
        raise ValueError(
            "Missing mktrf after t+1 market merge."
        )

    fit = sm.OLS(
        spread["P5-P1"],
        sm.add_constant(
            spread[["mktrf"]]
        ),
    ).fit()

    return {
        "monthly_alpha":
            float(
                fit.params["const"]
            ),
        "alpha_t_stat":
            float(
                fit.tvalues["const"]
            ),
        "market_beta":
            float(
                fit.params["mktrf"]
            ),
        "r_squared":
            float(
                fit.rsquared
            ),
        "months":
            int(
                fit.nobs
            ),
        "merged_data":
            spread,
        "fit":
            fit,
    }


In [ ]:
# A.6.16 Calculate required economic-performance statistics

rf_validation_return = mean_return_stats(
    rf_validation_portfolios["P5-P1"]
)

rf_test_return = mean_return_stats(
    rf_test_portfolios["P5-P1"]
)

ols_test_return = mean_return_stats(
    ols_test_portfolios["P5-P1"]
)

rf_validation_alpha = market_model_alpha(
    rf_validation_portfolios
)

rf_test_alpha = market_model_alpha(
    rf_test_portfolios
)

ols_test_alpha = market_model_alpha(
    ols_test_portfolios
)


In [ ]:
# A.6.17 Required A.6 results table

a6_results = pd.DataFrame([
    {
        "Model": "Random Forest",
        "Period": "Validation",
        "Months":
            rf_validation_return["months"],
        "Mean monthly P5-P1":
            rf_validation_return[
                "mean_p5_minus_p1"
            ],
        "P5-P1 t-stat":
            rf_validation_return[
                "mean_return_t_stat"
            ],
        "Monthly alpha":
            rf_validation_alpha[
                "monthly_alpha"
            ],
        "Alpha t-stat":
            rf_validation_alpha[
                "alpha_t_stat"
            ],
    },
    {
        "Model": "Random Forest",
        "Period": "Test",
        "Months":
            rf_test_return["months"],
        "Mean monthly P5-P1":
            rf_test_return[
                "mean_p5_minus_p1"
            ],
        "P5-P1 t-stat":
            rf_test_return[
                "mean_return_t_stat"
            ],
        "Monthly alpha":
            rf_test_alpha[
                "monthly_alpha"
            ],
        "Alpha t-stat":
            rf_test_alpha[
                "alpha_t_stat"
            ],
    },
    {
        "Model": "Pooled OLS",
        "Period": "Test",
        "Months":
            ols_test_return["months"],
        "Mean monthly P5-P1":
            ols_test_return[
                "mean_p5_minus_p1"
            ],
        "P5-P1 t-stat":
            ols_test_return[
                "mean_return_t_stat"
            ],
        "Monthly alpha":
            ols_test_alpha[
                "monthly_alpha"
            ],
        "Alpha t-stat":
            ols_test_alpha[
                "alpha_t_stat"
            ],
    },
])

a6_results


In [ ]:
# A.6.18 Percentage-format presentation table

a6_results_display = (
    a6_results.copy()
)

a6_results_display[
    "Mean monthly P5-P1 (%)"
] = (
    100
    * a6_results_display[
        "Mean monthly P5-P1"
    ]
)

a6_results_display[
    "Monthly alpha (%)"
] = (
    100
    * a6_results_display[
        "Monthly alpha"
    ]
)

a6_results_display = (
    a6_results_display[
        [
            "Model",
            "Period",
            "Months",
            "Mean monthly P5-P1 (%)",
            "P5-P1 t-stat",
            "Monthly alpha (%)",
            "Alpha t-stat",
        ]
    ]
)

a6_results_display.round(4)


In [ ]:
# A.6.19 Market-date alignment audit

market_alignment_audit = pd.DataFrame({
    "Series": [
        "RF Validation",
        "RF Test",
        "OLS Test",
    ],
    "Decision first month": [
        rf_validation_alpha[
            "merged_data"
        ]["decision_month"].min(),
        rf_test_alpha[
            "merged_data"
        ]["decision_month"].min(),
        ols_test_alpha[
            "merged_data"
        ]["decision_month"].min(),
    ],
    "Decision last month": [
        rf_validation_alpha[
            "merged_data"
        ]["decision_month"].max(),
        rf_test_alpha[
            "merged_data"
        ]["decision_month"].max(),
        ols_test_alpha[
            "merged_data"
        ]["decision_month"].max(),
    ],
    "Holding first month": [
        rf_validation_alpha[
            "merged_data"
        ]["holding_month"].min(),
        rf_test_alpha[
            "merged_data"
        ]["holding_month"].min(),
        ols_test_alpha[
            "merged_data"
        ]["holding_month"].min(),
    ],
    "Holding last month": [
        rf_validation_alpha[
            "merged_data"
        ]["holding_month"].max(),
        rf_test_alpha[
            "merged_data"
        ]["holding_month"].max(),
        ols_test_alpha[
            "merged_data"
        ]["holding_month"].max(),
    ],
    "Missing mktrf": [
        int(
            rf_validation_alpha[
                "merged_data"
            ]["mktrf"].isna().sum()
        ),
        int(
            rf_test_alpha[
                "merged_data"
            ]["mktrf"].isna().sum()
        ),
        int(
            ols_test_alpha[
                "merged_data"
            ]["mktrf"].isna().sum()
        ),
    ],
})

market_alignment_audit


In [ ]:
# A.6.20 Final protocol audit

print("A.6 ECONOMIC PERFORMANCE AUDIT")
print("-" * 55)

print(
    "Exact assignment quintile formula used:",
    True,
)

print(
    "Equal-weighted realised quintile returns:",
    True,
)

print(
    "RF validation months:",
    rf_validation_return["months"],
)

print(
    "RF test months:",
    rf_test_return["months"],
)

print(
    "OLS test months:",
    ols_test_return["months"],
)

print(
    "\nSame RF and OLS test stock-month rows:",
    rf_test_keys == ols_test_keys,
)

print(
    "Market return matched at t+1:",
    True,
)

print(
    "Missing market returns after matching:",
    (
        int(
            market_alignment_audit[
                "Missing mktrf"
            ].sum()
        )
    ),
)


## A.6 Summary

A.6 evaluates whether model forecasts translate into economically useful stock selection.

The same monthly prediction-sorted quintile rule is applied to Random Forest and pooled OLS. P1 contains the lowest-ranked stocks, P5 the highest-ranked stocks, and the long-short spread is the equal-weighted realised return on P5 minus P1.

For each required model-period combination, the notebook reports:

- average monthly P5−P1 excess return;
- t-statistic of the average monthly spread;
- monthly market-model alpha;
- alpha t-statistic.

Market attribution uses `mktrf` from the holding month \(t+1\), not the portfolio-formation month \(t\).
